# معمل إعداد بيئة عمل LangGraph باستخدام مدير الحزم السريع UV
### Interactive Hands-on Lab: Setting up LangGraph with UV in VS Code (Lecture 69)

أهلاً بك يا بطل!
في المحاضرة تم شرح استخدام Conda، لكن في هذا المعمل سنعتمد على المعيار الحديث الأسرع عالمياً: **مدير الحزم `uv`** المكتوب بلغة Rust.
ستتعلم كيف تجهز بيئة عمل كاملة في ثوانٍ معدودة بدلاً من الانتظار لعدة دقائق مع Conda أو pip القديم.

الأوامر الأساسية التي نستخدمها في سطر الأوامر (Terminal):
```bash
# 1. إنشاء بيئة بايثون 3.12 معزولة في أقل من ثانية:
uv venv .venv --python 3.12

# 2. تثبيت الحزم الأساسية بسرعة خارقة:
uv pip install -r requirements.txt

# 3. تثبيت نواة jupyter لتعمل داخل VS Code:
uv pip install ipykernel
```

سنتحقق في هذا الدفتر عملياً من:
1. تشغيل بايثون داخل البيئة المعزولة `.venv` المنشأة بـ `uv`.
2. التأكد من توفر مكتبات LangGraph الأساسية وحزمة `ipykernel`.
3. فحص ملف المتغيرات السرية (`.env`) بأمان.
4. التعرف على خدمة المراقبة LangSmith.
5. تشغيل جراف فحص سريع ومبسط.
6. قسم خاص: جرب بنفسك!


In [1]:
import sys
import os

print("مسار مفسر بايثون داخل بيئة uv الافتراضية:")
print(sys.executable)
print("\nإصدار بايثون المستخدم:")
print(sys.version.split()[0])


مسار مفسر بايثون داخل بيئة uv الافتراضية:
D:\Code\AI\rag-mastery\.venv\Scripts\python.exe

إصدار بايثون المستخدم:
3.12.12


### 1. فحص تثبيت المكتبات الأساسية المثبتة بـ UV
تعال نتأكد من أن المكتبات الأساسية (langchain, langgraph, ipykernel) التي تم تثبيتها بواسطة `uv pip` موجودة وتعمل بكفاءة.


In [2]:
try:
    import langgraph
    import langchain_core
    import ipykernel
    print("كل مكتبات LangGraph وحزمة ipykernel مثبتة وجاهزة للعمل بنجاح!")
except ImportError as error:
    print("تنبيه: هناك مكتبة ناقصة:", error)


كل مكتبات LangGraph وحزمة ipykernel مثبتة وجاهزة للعمل بنجاح!


### 2. فحص ملف المتغيرات السرية (.env)
ملف `.env` هو المكان الآمن الذي نضع فيه مفاتيحنا حتى لا تتسرب في الكود.
تعال نفحص هل المفاتيح متوفرة في بيئتنا دون طباعة نص المفتاح نفسه.


In [3]:
import dotenv

dotenv.load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

print("حالة المفاتيح السرية في ملف .env:")
print("- مفتاح Groq:", "متوفر وجاهز" if groq_key else "غير موجود")
print("- مفتاح OpenAI:", "متوفر وجاهز" if openai_key else "غير موجود")


حالة المفاتيح السرية في ملف .env:
- مفتاح Groq: متوفر وجاهز
- مفتاح OpenAI: غير موجود


### 3. ما هو LangSmith ولماذا نحتاجه؟
في تطبيقات الوكلاء، قد يدور الوكيل في حلقات تفكير واستدعاء أدوات متعددة.
خدمة LangSmith تسمح لنا بمشاهدة كل خطوة وتكلفتها والوقت المستغرق بالملي ثانية.


In [4]:
tracing = os.getenv("LANGCHAIN_TRACING_V2", "false")
project = os.getenv("LANGCHAIN_PROJECT", "rag-mastery-langgraph")

print(f"حالة تفعيل التتبع (LangSmith Tracing): {tracing}")
print(f"اسم المشروع المسجل: {project}")


حالة تفعيل التتبع (LangSmith Tracing): false
اسم المشروع المسجل: rag-mastery-langgraph


### 4. تجربة تشغيل جراف فحص سريع (Smoke Test)
تعال نبني جرافاً صغيراً جداً مهمته فحص جاهزية بيئة عمل UV والتأكد من أن محرك LangGraph يعمل بدون أخطاء.


In [5]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class QuickCheckState(TypedDict):
    status: str
    checks: List[str]

def step_one(state: QuickCheckState):
    print("تنفيذ الخطوة الأولى: فحص محرك LangGraph وبيئة UV...")
    return {
        "status": "in_progress",
        "checks": state["checks"] + ["UV Environment & Engine: OK"]
    }

def step_two(state: QuickCheckState):
    print("تنفيذ الخطوة الثانية: فحص نواة ipykernel والذاكرة...")
    return {
        "status": "ready",
        "checks": state["checks"] + ["Jupyter Kernel & Memory: OK"]
    }

# بناء الجراف
builder = StateGraph(QuickCheckState)
builder.add_node("check_engine", step_one)
builder.add_node("check_kernel", step_two)

builder.add_edge(START, "check_engine")
builder.add_edge("check_engine", "check_kernel")
builder.add_edge("check_kernel", END)

app = builder.compile()

# تشغيل الجراف
result = app.invoke({"status": "starting", "checks": []})
print("\nالنتيجة النهائية لفحص بيئة UV:")
print(f"الحالة: {result['status']}")
print("سجل الفحوصات:", result["checks"])


تنفيذ الخطوة الأولى: فحص محرك LangGraph وبيئة UV...
تنفيذ الخطوة الثانية: فحص نواة ipykernel والذاكرة...

النتيجة النهائية لفحص بيئة UV:
الحالة: ready
سجل الفحوصات: ['UV Environment & Engine: OK', 'Jupyter Kernel & Memory: OK']


### 5. جرب بنفسك! (Hands-on Experiments)
1. **تحدي 1**: افتح الطرفية (Terminal) وجرب كتابة الأمر `uv pip list`، وقارن سرعة الاستجابة بما كنت تراه في Conda!
2. **تحدي 2**: أضف عقدة ثالثة في الجراف باسم `check_provider` تفحص هل مفتاح Groq موجود وتضيف ذلك لسجل الفحوصات.
3. **تحدي 3**: جرب إضافة متغير جديد في الـ State مثل `package_manager: str` واجعل قيمته 'uv' وشاهد كيف يتم تمريرها حتى نهاية الجراف.
